# Value-only polynomial-zonotope L² benchmarks

This notebook isolates the zero-jet path: it propagates only the certified function-value polynomial zonotope and directly integrates its squared Euclidean norm. Jacobian and Hessian enclosures are never allocated.

The main target is a scalar-output tanh network with input dimension 100 and three hidden layers of width 50. Timings are deliberately split into value propagation, direct squared integration, and the public adaptive `pz_l2norm` API.

In [ ]:
from collections import Counter
from time import perf_counter

import pandas as pd
import torch

from intervalnets import (
    IntervalTensor,
    PZIntegrationCell,
    enable_interval_eval,
    integrate_pz_value_squared,
)

torch.set_num_threads(1)
torch.manual_seed(20260731)
enable_interval_eval()
print({'torch': torch.__version__, 'threads': torch.get_num_threads()})

In [ ]:
def make_model(input_dim, hidden_widths, output_dim=1, seed=20260731):
    torch.manual_seed(seed)
    layers = []
    previous = input_dim
    for width in hidden_widths:
        layers.extend([
            torch.nn.Linear(previous, width, dtype=torch.float64),
            torch.nn.Tanh(),
        ])
        previous = width
    layers.append(torch.nn.Linear(previous, output_dim, dtype=torch.float64))
    return torch.nn.Sequential(*layers)


def benchmark_case(input_dim, hidden_widths, *, half_width=0.1, iterations=0, seed=20260731):
    model = make_model(input_dim, hidden_widths, seed=seed)
    box = IntervalTensor.from_bounds(
        [-half_width] * input_dim,
        [half_width] * input_dim,
    )
    cell = PZIntegrationCell.from_affine_box(box)

    start = perf_counter()
    traced = model.eval_pz_value(cell.domain, return_trace=True)
    forward_s = perf_counter() - start

    start = perf_counter()
    squared = integrate_pz_value_squared(traced.final, cell)
    integration_s = perf_counter() - start

    start = perf_counter()
    norm = model.pz_l2norm(box, iterations=iterations)
    public_s = perf_counter() - start

    kinds = Counter(traced.final.noise_kinds)
    row = {
        'architecture': f"{input_dim}-{'-'.join(map(str, hidden_widths))}-1",
        'iterations': iterations,
        'forward_s': forward_s,
        'integration_s': integration_s,
        'manual_total_s': forward_s + integration_s,
        'public_norm_s': public_s,
        'terms': len(traced.final.terms),
        'num_noise': traced.final.num_noise,
        'domain_noise': kinds['domain'],
        'pointwise_noise': kinds['approximation_pointwise'],
        'max_degree': max((sum(exp) for exp in traced.final.terms), default=0),
        'norm_lower': float(norm.lower),
        'norm_upper': float(norm.upper),
        'norm_width': float(norm.upper) - float(norm.lower),
    }
    layer_rows = [
        {
            'layer_index': record.layer_index,
            'layer': record.layer_type,
            'terms': record.summary['term_count'],
            'noise': record.summary['num_noise'],
            'degree': record.summary['max_degree'],
        }
        for record in traced.records
    ]
    return row, pd.DataFrame(layer_rows), model, box

## Architecture sweep

The final affine support should contain `input_dim + sum(hidden_widths)` terms: one domain symbol per input coordinate and one pointwise residual symbol per hidden neuron. Degree should remain one.

In [ ]:
architectures = [
    (2, [5]),
    (10, [20]),
    (25, [50]),
    (50, [50, 50]),
    (100, [50, 50, 50]),
]

rows = [benchmark_case(input_dim, hidden)[0] for input_dim, hidden in architectures]
architecture_results = pd.DataFrame(rows)
architecture_results

In [ ]:
largest, largest_layers, largest_model, largest_box = benchmark_case(100, [50, 50, 50])
assert largest['terms'] == 100 + 3 * 50
assert largest['domain_noise'] == 100
assert largest['pointwise_noise'] == 3 * 50
assert largest['max_degree'] == 1
assert largest['public_norm_s'] < 3.0, largest
display(pd.Series(largest, name='largest_case'))
largest_layers

## Adaptive cost and enclosure quality

Each marked cell is replaced by two children. With one marked root, iteration 1 therefore evaluates three cells in total (the discarded parent plus two children). The enclosure width is also sensitive to input-box size: affine residuals grow with wider preactivation intervals even when runtime changes little.

In [ ]:
adaptive_rows = [
    benchmark_case(100, [50, 50, 50], iterations=iterations)[0]
    for iterations in (0, 1, 2)
]
pd.DataFrame(adaptive_rows)[['iterations', 'public_norm_s', 'norm_lower', 'norm_upper', 'norm_width']]

In [ ]:
scale_rows = [
    benchmark_case(100, [50, 50, 50], half_width=half_width)[0]
    for half_width in (0.01, 0.05, 0.1, 0.25, 0.5, 1.0)
]
pd.DataFrame(scale_rows)[['architecture', 'public_norm_s', 'norm_lower', 'norm_upper', 'norm_width']].assign(
    half_width=(0.01, 0.05, 0.1, 0.25, 0.5, 1.0)
)

## Correctness spot check against the two-jet path

The zero-jet uses fewer noise dimensions because it never introduces the derivative and second-derivative residual symbols. On a small network, its function-value interval should nevertheless match the `Y` component of the full two-jet.

In [ ]:
small_model = make_model(2, [5])
small_box = IntervalTensor.from_bounds([-0.1, -0.1], [0.1, 0.1])
small_cell = PZIntegrationCell.from_affine_box(small_box)
value = small_model.eval_pz_value(small_cell.domain)
jet = small_model.eval_pz_twojet(small_cell.domain)
value_interval = value.interval_enclosure()
jet_interval = jet.Y.interval_enclosure()

assert torch.allclose(torch.tensor(value_interval.lower), torch.tensor(jet_interval.lower))
assert torch.allclose(torch.tensor(value_interval.upper), torch.tensor(jet_interval.upper))
pd.Series({
    'value_terms': len(value.terms),
    'twojet_Y_terms': len(jet.Y.terms),
    'value_noise': value.num_noise,
    'twojet_noise': jet.Y.num_noise,
})

## Reading the diagnostics

- Runtime stays low because affine activation enclosures preserve degree one and add terms only linearly: `input_dim + total_hidden_neurons`.
- Direct L² integration recognizes unit-vector affine support and evaluates the current certified pointwise-residual semantics without constructing pair-exponent rows or a squared PZ.
- The remaining forward bottleneck is repeated canonical exponent validation and reconstruction inside `PolynomialZonotope`; this is the first target if much wider/deeper zero-jets are needed.
- Runtime and enclosure quality are different questions. Wide high-dimensional boxes can make affine residual uncertainty dominate and drive the certified lower L² bound to zero even though the computation is fast. Domain partitioning or tighter/non-affine activation enclosures address tightness, not this runtime bottleneck.